In [1]:
!pip install -q datasets
!pip install -q scikit-learn
!pip install -q lime
!pip install -q fairlearn
!pip install joblib
!pip install transformers
!pip install torch
!pip install sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-c

In [2]:
import numpy as np
import pandas as pd
import re
import logging
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from lime.lime_text import LimeTextExplainer
import joblib
from fairlearn.metrics import demographic_parity_difference
import warnings

# Configure logging
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

# Suppress scikit-learn warnings
warnings.filterwarnings('ignore')

# Enhanced privacy-preserving text processing
PII_PATTERNS = {
    'email': r'\S+@\S+',
    'phone': r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b',
    'credit_card': r'\b\d{4} ?\d{4} ?\d{4} ?\d{4}\b',
    'ssn': r'\b\d{3}-\d{2}-\d{4}\b'
}

def anonymize_text(text):
    """Remove PII from text with enhanced pattern matching"""
    for key, pattern in PII_PATTERNS.items():
        text = re.sub(pattern, f'[{key.upper()}]', text)
    return text

# Ethical response templates
RESPONSE_TEMPLATES = {
    'support': "I'm sorry to hear you're experiencing this issue. Let me help you with that. Could you please provide more details?",
    'positive': "Thank you for your feedback! We're happy to hear you're satisfied with our service.",
    'other': "I want to ensure I understand correctly. Could you please rephrase your question?",
    'escalation': "Let me connect you with a human representative for further assistance.",
    'privacy': "I'm unable to process personal information. Please remove any sensitive data and try again."
}

class EthicalChatbot:
    def __init__(self, model, vectorizer, confidence_threshold=0.7):
        self.model = model
        self.vectorizer = vectorizer
        self.confidence_threshold = confidence_threshold
        self.explainer = LimeTextExplainer(class_names=model.classes_)

    def predict_ethical_response(self, text):
        """Generate an ethical response with guardrails"""
        # Privacy check
        if any(re.search(pattern, text) for pattern in PII_PATTERNS.values()):
            return RESPONSE_TEMPLATES['privacy'], None, None

        # Anonymize input
        clean_text = anonymize_text(text)

        # Model prediction
        vec_text = self.vectorizer.transform([clean_text])
        probabilities = self.model.predict_proba(vec_text)[0]
        pred_class = self.model.predict(vec_text)[0]
        confidence = np.max(probabilities)

        # Ethical guardrails
        if confidence < self.confidence_threshold:
            return RESPONSE_TEMPLATES['escalation'], pred_class, confidence

        # Bias mitigation check
        if self._check_potential_bias(clean_text):
            return RESPONSE_TEMPLATES['escalation'], pred_class, confidence

        return RESPONSE_TEMPLATES.get(pred_class, RESPONSE_TEMPLATES['other']), pred_class, confidence

    def _check_potential_bias(self, text):
        """Simple bias detection using sensitive term list"""
        sensitive_terms = ['gender', 'race', 'religion', 'ethnicity']
        return any(term in text.lower() for term in sensitive_terms)

    def explain_prediction(self, text):
        """Provide LIME explanation for predictions"""
        vec_text = self.vectorizer.transform([text])
        exp = self.explainer.explain_instance(
            text,
            lambda x: self.model.predict_proba(self.vectorizer.transform(x)),
            num_features=10
        )
        return exp.as_list()

# Enhanced data processing
def load_and_process_data():
    """Load and process data with balanced sampling"""
    try:
        ds = load_dataset("TNE-AI/customer-support-on-twitter-conversation")
        processed_data = []

        for conv in ds['train']:
            text = conv.get('conversation', '')
            label = 'other'

            # Expanded keyword sets
            positive_kws = ['thank', 'appreciate', 'good', 'great', 'love', 'happy', 'satisfied', 'awesome', 'excellent']
            support_kws = [
                'problem', 'issue', 'help', 'fix', 'error', 'trouble', 'support', 'login', 'log in',
                'can’t', 'cannot', 'reset', 'password', 'access', 'fail', 'locked'
            ]

            # Score keyword occurrences
            text_lower = text.lower()
            positive_score = sum(1 for kw in positive_kws if kw in text_lower)
            support_score = sum(1 for kw in support_kws if kw in text_lower)

            # Improved labeling logic
            if support_score >= positive_score and support_score > 0:
                label = 'support'
            elif positive_score > 0:
                label = 'positive'
            else:
                label = 'other'

            processed_data.append({
                'text': anonymize_text(text),
                'label': label
            })

        df = pd.DataFrame(processed_data)

        # Balance classes
        min_samples = min(df['label'].value_counts().values)
        df = df.groupby('label').apply(lambda x: x.sample(min_samples)).reset_index(drop=True)

        return df

    except Exception as e:
        logging.error("Error processing dataset: %s", str(e))
        raise


# Enhanced evaluation
def evaluate_model(y_true, y_pred, sensitive_attributes=None):
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'f1': f1_score(y_true, y_pred, average='weighted', zero_division=0)
    }

    if sensitive_attributes is not None:
        metrics['demographic_parity_diff'] = demographic_parity_difference(
            y_true, y_pred, sensitive_features=sensitive_attributes
        )

    return metrics

# Interactive chat demo
def chat_demo(chatbot):
    print("\nChatbot: Hello! How can I assist you today? (Type 'exit' to end)")
    while True:
        user_input = input("\nUser: ")
        if user_input.lower() == 'exit':
            break

        response, pred_class, confidence = chatbot.predict_ethical_response(user_input)
        print(f"\nChatbot: {response}")
        print(f"Debug - Predicted: {pred_class}, Confidence: {confidence:.2f}")

        if confidence < chatbot.confidence_threshold:
            explanation = chatbot.explain_prediction(user_input)
            print("\nExplanation (Low confidence):")
            for feature, weight in explanation:
                print(f"{feature}: {weight:.4f}")

# Main workflow
if __name__ == "__main__":
    try:
        logging.info("Starting data loading and processing...")
        df = load_and_process_data()
        logging.info("Final label distribution:\n%s", df['label'].value_counts())

        # Split dataset
        logging.info("Splitting dataset into training and testing sets...")
        X_train, X_test, y_train, y_test = train_test_split(
            df['text'],
            df['label'],
            test_size=0.2,
            random_state=42,
            stratify=df['label']
        )

        # Text vectorization
        logging.info("Vectorizing text data...")
        vectorizer = TfidfVectorizer(
            max_features=5000,
            stop_words='english',
            ngram_range=(1, 2)
        )
        X_train_vec = vectorizer.fit_transform(X_train)
        X_test_vec = vectorizer.transform(X_test)

        # Train model
        logging.info("Training the Logistic Regression model...")
        model = LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            solver='lbfgs',
            random_state=42
        )
        model.fit(X_train_vec, y_train)

        # Create chatbot instance
        chatbot = EthicalChatbot(model, vectorizer)

        # Evaluate performance
        logging.info("Evaluating model performance...")
        y_pred = model.predict(X_test_vec)
        metrics = evaluate_model(y_test, y_pred)
        logging.info("Classification metrics:\n%s", pd.Series(metrics))

        # Start interactive demo
        chat_demo(chatbot )

        # Save model artifacts
        logging.info("Saving model artifacts...")
        joblib.dump(model, "ethical_chatbot_model.pkl")
        joblib.dump(vectorizer, "ethical_chatbot_vectorizer.pkl")
        logging.info("Model artifacts saved successfully.")

    except Exception as e:
        logging.error("Workflow failed: %s", str(e))


README.md:   0%|          | 0.00/430 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/217M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/794335 [00:00<?, ? examples/s]


Chatbot: Hello! How can I assist you today? (Type 'exit' to end)

User: I can't access my account

Chatbot: Let me connect you with a human representative for further assistance.
Debug - Predicted: support, Confidence: 0.66

Explanation (Low confidence):
access: -0.0140
account: 0.0053
my: 0.0004
t: 0.0004
can: 0.0003
I: 0.0003

User: this box is sealed tight and hard to open

Chatbot: I want to ensure I understand correctly. Could you please rephrase your question?
Debug - Predicted: other, Confidence: 0.84

User: exit


In [ ]:
import nbformat

# Set the full path to your notebook in Google Drive
file_path = '/content/drive/MyDrive/Colab Notebooks/AI_Chatbot.ipynb'

# Mount Google Drive (if not already mounted)
from google.colab import drive
drive.mount('/content/drive')

# Read the notebook
with open(file_path, 'r', encoding='utf-8') as f:
    nb = nbformat.read(f, as_version=4)

# Remove problematic metadata
if 'widgets' in nb['metadata']:
    del nb['metadata']['widgets']

# Optionally strip cell outputs (comment out if not needed)
for cell in nb['cells']:
    if 'outputs' in cell:
        cell['outputs'] = []
    if 'execution_count' in cell:
        cell['execution_count'] = None

# Save the cleaned notebook
cleaned_path = '/content/AI_Chatbot_CLEANED.ipynb'
with open(cleaned_path, 'w', encoding='utf-8') as f:
    nbformat.write(nb, f)

print(" Notebook cleaned and saved to", cleaned_path)
